# Cleaning and Reporting Retail Sales

*Clean supplied orders, attach a lookup table, and verify grouped and pivoted revenue reports.*


## Before You Run

In **Add files**, select `retail_sales_raw.csv` from this lesson's supplied assets.
Enter `assets/09_retail_sales` for **Save folder**. The upload creates
missing folders and places the file at
`/home/pyodide/assets/09_retail_sales/retail_sales_raw.csv`.

`/home/pyodide` is the browser's virtual filesystem, not a folder on your
computer. The upload folder is relative to `/home/pyodide`; Python relative
paths use the current directory returned by `os.getcwd()`. Keep the default
working directory in the course site. Locally, run from this chapter folder.
Uploading again to the same complete path replaces that input file.
Saved examples appear under `outputs/09_retail_sales/` in the file browser.


## Raw Orders

The supplied file has 12 rows and 9 fields: order ID, date, region,
category, product name, quantity, unit price, customer type, and channel.
Text case, spaces, currency symbols, and date formats vary. Order 1006
is repeated, and order 1005 lacks a quantity.

This case represents each order once. Cleaning retains the first of
the identical order 1006 records and estimates the missing quantity
using the median of the retained orders. Revenue is quantity multiplied
by unit price, in dollars.

A prepared lookup assigns Seoul and Incheon to the **Capital** service
team, and Busan and Daegu to the **Regional** service team. The report
summarizes revenue by region and product category using the raw input;
the separate 15-row exploration file is a different dataset.


## Imports, Paths, and Lookup Data

The small lookup is defined here so this notebook can run independently.
Give generated order and report files distinct names in the same output folder.


In [1]:
import os

import pandas as pd

input_directory = os.path.join("assets", "09_retail_sales")
output_directory = os.path.join("outputs", "09_retail_sales")
input_path = os.path.join(input_directory, "retail_sales_raw.csv")
orders_output_path = os.path.join(output_directory, "cleaned_orders.csv")
report_output_path = os.path.join(output_directory, "regional_revenue.csv")
os.makedirs(output_directory, exist_ok=True)

service_teams = pd.DataFrame({
    "region": ["Busan", "Daegu", "Incheon", "Seoul"],
    "service_team": ["Regional", "Regional", "Capital", "Capital"],
})


## Normalize and Retain One Record per Order

Work on a copy and use the same conversions as the guided lessons.
After discarding the duplicate, compute the median and record which
quantities were imputed in a Boolean column. This keeps the estimate
distinguishable from observed values in the saved file.


In [2]:
def clean_orders(raw):
    orders = raw.copy()
    orders.columns = orders.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    for column in ["region", "category", "customer_type", "channel"]:
        orders[column] = orders[column].str.strip().str.title()
    orders["product_name"] = orders["product_name"].str.strip()
    orders["unit_price"] = pd.to_numeric(
        orders["unit_price"].str.replace("$", "", regex=False), errors="coerce"
    )
    orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
    orders["order_date"] = pd.to_datetime(orders["order_date"], format="mixed", errors="coerce")
    orders = orders.drop_duplicates(subset=["order_id"], keep="first").copy()

    orders["quantity_imputed"] = orders["quantity"].isna()
    median_quantity = orders["quantity"].median()
    orders["quantity"] = orders["quantity"].fillna(median_quantity).astype(int)
    orders["revenue"] = orders["quantity"] * orders["unit_price"]
    orders = orders.sort_values(["order_date", "order_id"]).reset_index(drop=True)
    return orders, median_quantity


## Summarize and Attach the Regional Lookup

Named aggregation provides an order count and revenue total per region.
The left merge keeps all regions and checks that the lookup has one record
per region. The pivot gives a second view of exactly the same order revenue.


In [3]:
def build_reports(orders, teams):
    regional = orders.groupby("region", as_index=False, sort=True).agg(
        order_count=("order_id", "size"),
        total_revenue=("revenue", "sum"),
    )
    regional = pd.merge(regional, teams, on="region", how="left", validate="many_to_one")
    pivot = orders.pivot_table(
        values="revenue", index="region", columns="category",
        aggfunc="sum", fill_value=0, margins=True,
    )
    return regional, pivot


## Save and Reopen Both Results

The orders file retains the imputation flag. Restore its date type on
reload; the regional report contains only text and numeric fields.
Both outputs replace fixed filenames when the workflow runs again.


In [4]:
def save_reports(orders, regional):
    orders.to_csv(orders_output_path, index=False, encoding="utf-8", date_format="%Y-%m-%d")
    regional.to_csv(report_output_path, index=False, encoding="utf-8")
    reopened_orders = pd.read_csv(orders_output_path, encoding="utf-8", parse_dates=["order_date"])
    reopened_regional = pd.read_csv(report_output_path, encoding="utf-8")
    return reopened_orders, reopened_regional


## Run the Workflow and Check the Totals

Display the raw preview before cleaning. Report the retained order count,
estimated quantity, and two revenue summaries. Check that every retained
order contributes once and that both saved tables match their in-memory results.


In [5]:
def main():
    raw = pd.read_csv(input_path, encoding="utf-8")
    print("Raw preview:")
    print(raw.head(3).to_string(index=False))
    print("Raw row count:", len(raw))
    orders, median_quantity = clean_orders(raw)
    print("Retained order count:", len(orders))
    print("Median used for missing quantity:", median_quantity)
    print("Imputed orders:")
    print(orders.loc[orders["quantity_imputed"], ["order_id", "quantity", "revenue"]])
    print("Retained quantity summary:")
    print(orders["quantity"].describe().round(2))

    regional, pivot = build_reports(orders, service_teams)
    print("Regional revenue (dollars):")
    print(regional)
    print("Revenue pivot (dollars):")
    print(pivot)
    grand_total = orders["revenue"].sum()
    print("Grand total:", grand_total)
    print("Totals agree:", grand_total == regional["total_revenue"].sum() == pivot.loc["All", "All"])

    reopened_orders, reopened_regional = save_reports(orders, regional)
    print("Saved files:", orders_output_path, report_output_path)
    print("Saved orders match:", reopened_orders.equals(orders))
    print("Saved regional report matches:", reopened_regional.equals(regional))
    print("Reopened order count:", len(reopened_orders))
    return orders, regional, pivot


cleaned_orders_df, regional_report_df, revenue_pivot = main()


Raw preview:
 Order ID Order Date Region   Category Product Name  Quantity Unit Price Customer Type Channel
     1001 2026/01/03  seoul     laptop Notebook Pro       2.0      $1200     corporate  online
     1002 2026/01/05   Busan accessory        Mouse       5.0        $25    individual offline
     1003 2026/01/07   SEOUL   monitor UltraView 27       1.0       $310     corporate  online
Raw row count: 12
Retained order count: 11
Median used for missing quantity: 2.0
Imputed orders:
   order_id  quantity  revenue
4      1005         2       90
Retained quantity summary:
count    11.00
mean      2.18
std       1.17
min       1.00
25%       1.50
50%       2.00
75%       2.50
max       5.00
Name: quantity, dtype: float64
Regional revenue (dollars):
    region  order_count  total_revenue service_team
0    Busan            3            880     Regional
1    Daegu            2            370     Regional
2  Incheon            2           2180      Capital
3    Seoul            4           

## Result Analysis

Cleaning reduces **12 raw rows to 11 unique orders**.
The retained orders have median quantity **2**, used for order 1005.
Its estimated revenue is **90 dollars**, and `quantity_imputed` records
that assumption. The revenue total therefore includes one estimated quantity.

Total revenue is **8,175 dollars**: Busan 880, Daegu 370, Incheon 2,180,
and Seoul 4,745. Category totals are Accessory 705, Laptop 6,540, and
Monitor 930. The regional and pivot totals agree with the order total.

Both saved-table comparisons are `True`. The orders file retains 11 rows
and the regional file retains 4 rows. Repeated runs reload the raw input
and replace the same files, preserving the supplied data.

These results differ from the separate 15-order exploration file because
that file has extra orders and a different quantity for order 1005.
